# V036: cold grouped confirmation of V035

Seven freshly fitted members, fixed A-selected budgets and one frozen winner. Two ordinary-lease stages; no F label conversion until all predictions are locked. Original A-validation excluded. Default inert.

In [ ]:
import os, json, hashlib, copy, gc, math, fcntl, time
from pathlib import Path
from datetime import datetime, timezone
import numpy as np
import pandas as pd
V036_PROTOCOL_SHA='e9fb1fc5fd7a342fd744f8ed2d7bcdbb4eab3d3e5be8f12da6f93575374ab0f4'
V036_V013_SOURCE_SHA='56cb9d7e824e58126bddd71c7c0f7063844974c31ee00a6dac23f2112657615a'
V036_NOTEBOOK='confirmation_v036.ipynb'
V036_OUT='outputs/confirmation_v036'

def v036_require(c,message):
    if not bool(c):raise ValueError(message)
def v036_code(p):
    n=json.loads(Path(p).read_text());return '\n'.join(''.join(c['source']) for c in n['cells'] if c['cell_type']=='code')
def v036_sha(p):return hashlib.sha256(Path(p).read_bytes()).hexdigest()
def v036_sig(v):return hashlib.sha256(json.dumps(v,sort_keys=True,separators=(',',':'),allow_nan=False).encode()).hexdigest()
def v036_immutable_json(p,v,resume):
    if p.exists():v036_require(resume and json.loads(p.read_text())==v,'Immutable file changed: '+str(p))
    else:v013_json(p,v)
def v036_immutable_array(p,v):
    if p.exists():v036_require(np.array_equal(np.load(p,allow_pickle=False),v),'Existing array differs: '+str(p))
    else:v013_array(p,v)
def v036_memory_factory(spec,n_features,prep):return v8_memory_model(spec,n_features,prep)
def v036_offset_factory(spec,n_features,prep):return v9_offset_model(spec,n_features,prep)
def v036_plain_factory(spec,n_features,prep):return V5PressureRNN(spec,n_features)
for _factory in [v036_memory_factory,v036_offset_factory,v036_plain_factory]:_factory.__module__='confirmation_v036'
def v036_factory(name):
    if name=='memory':return v036_memory_factory
    if name=='offset':return v036_offset_factory
    if name in ['curve','deep_curve']:return v036_plain_factory
    if name=='capacity_laplace_curve':return v029_laplace_factory
    raise ValueError('Unknown V8 member: '+name)

def v036_metrics(p,y):
    e=np.abs(p-y);return dict(mae=float(e.mean()),median_absolute_error=float(np.median(e)),p90_absolute_error=float(np.quantile(e,.9)))

def v036_load_selected_labels(path,selected_breath_ids):
    """Filter before pressure conversion; never numerically load other targets."""
    import csv
    ids=np.asarray(selected_breath_ids,dtype=np.int64);where={int(b):i for i,b in enumerate(ids)};values=np.empty((len(ids),80),dtype=np.float32);counts=np.zeros(len(ids),dtype=np.int16)
    with Path(path).open(newline='') as f:
        for row in csv.DictReader(f):
            i=where.get(int(row['breath_id']))
            if i is None:continue
            j=int(counts[i]);v036_require(j<80,'Too many selected target rows');values[i,j]=np.float32(float(row['pressure']));counts[i]+=1
    v036_require(len(where)==len(ids) and np.all(counts==80) and np.isfinite(values).all(),'Selected target coverage/nonfinite')
    return values

def v036_verify_record(root,record,identity):
    v036_require(record['identity']==identity and record['identity_sha256']==v036_sig(identity),'Member identity changed')
    for key,hkey in [('checkpoint_file','checkpoint_sha256'),('last_checkpoint_file','last_checkpoint_sha256'),('report_file','report_sha256')]:v036_require(v036_sha(root/record[key])==record[hkey],'Completed F member changed: '+key)
    expected=Path(V036_OUT)/'F_models'/identity['member_id']
    v036_require(record['checkpoint_file']==str(expected/'model.pt') and record['report_file']==str(expected/'report.json') and record['last_checkpoint_file']==str(expected/'model.last.pt'),'F record must reference its own cold-run directory')
    report=json.loads((root/record['report_file']).read_text());recipe=identity['recipe']
    v036_require(report['completed'] and report['executed_epochs']==recipe['selected_epochs'] and report['training_breaths']==54323 and not report['validation_used'] and report['executed_lr_schedule']==recipe['lr_schedule'],'Completed F budget/config changed')
    return report


def v036_fit_dispatch(name,recipe,raw,pressure,phase,checkpoint,*,feature_names,device,resume,source_fingerprints,epoch_log_path=None,on_epoch=None):
    args=(raw,pressure,phase,recipe['model_spec'],recipe['train_cfg'],recipe['seed'],recipe['lr_schedule'],checkpoint)
    opts=dict(epoch_log_path=epoch_log_path,feature_names=feature_names,device=device,resume=resume,on_epoch=on_epoch)
    if name in ['deep','original_compact']:return fit_fixed(*args,**opts)
    if name=='capacity_laplace_curve':return fit_fixed_v029(*args,**opts,model_factory=v029_laplace_factory,loss_kind='laplace_curve',arm='v036_F_'+name,source_fingerprints=source_fingerprints)
    return fit_fixed_v008(*args,**opts,model_factory=v036_factory(name),loss_kind=recipe['loss_kind'],arm='v036_F_'+name,source_fingerprints=source_fingerprints)

def v036_predict_dispatch(name,recipe,checkpoint,raw,*,device,source_fingerprints,batch_size=512):
    if name in ['deep','original_compact']:return predict_checkpoint(checkpoint,raw,device=device,batch_size=batch_size)
    if name=='capacity_laplace_curve':return predict_checkpoint_v029(checkpoint,raw,device=device,batch_size=batch_size,model_factory=v029_laplace_factory,expected_arm='v036_F_'+name,expected_loss_kind='laplace_curve',source_fingerprints=source_fingerprints)
    return predict_checkpoint_v008(checkpoint,raw,device=device,batch_size=batch_size,model_factory=v036_factory(name),expected_arm='v036_F_'+name,expected_loss_kind=recipe['loss_kind'],source_fingerprints=source_fingerprints)

def run_confirmation_v036(root='.',resume=False,*,stage_key):
    root=Path(root).resolve();out=root/V036_OUT;out.mkdir(parents=True,exist_ok=True);locks=[];began=time.monotonic()
    try:
        for p in [root/'outputs/.v011_gpu.lock',out/'.run.lock']:
            f=p.open('a+');fcntl.flock(f,fcntl.LOCK_EX|fcntl.LOCK_NB);locks.append(f)
        v036_require(v036_sha(root/'research/confirmation_protocol_v036.json')==V036_PROTOCOL_SHA,'F protocol changed')
        plan=json.loads((root/'research/confirmation_protocol_v036.json').read_text())
        v036_require(stage_key in plan['stages'],'Unknown frozen execution stage')
        for p,h in plan['input_hashes'].items():v036_require(v036_sha(root/p)==h,'F prerequisite changed: '+p)
        v036_require(v036_sha(root/'data/train.csv')==plan['train_SHA256'],'Approved data changed')
        v036_require(v036_sha(root/plan['best_CSV'])==plan['best_CSV_SHA256'],'Accepted official CSV changed')
        helper=v036_code(root/'controlled_experiments_v013.ipynb')
        v036_require(hashlib.sha256(helper.encode()).hexdigest()==V036_V013_SOURCE_SHA,'Helper notebook changed')
        v036_require(os.environ.get('PA3_RUN_V013','0')!='1','Disable A-screen run flag')
        exec(compile(helper,'controlled_experiments_v013.ipynb','exec'),globals())
        source=hashlib.sha256(v036_code(root/V036_NOTEBOOK).encode()).hexdigest()
        v036_require(source==os.environ.get('PA3_V036_SOURCE_SHA256'),'F source differs from reviewed pin')
        for filename,h in plan['source_code_hashes'].items():
            code=v036_code(root/'research'/filename);v036_require(hashlib.sha256(code.encode()).hexdigest()==h,'Component changed: '+filename)
            exec(compile(code,filename,'exec'),globals())
        torch.set_num_threads(4);v036_require(torch.cuda.is_available(),'Actual approved CUDA required');seed_sequence(plan['seed'])
        v036_require(torch.are_deterministic_algorithms_enabled(),'Strict determinism required')
        proof=json.loads((out/'course_proofs.json').read_text())
        v036_require(proof['status']=='passed' and proof['actual_cuda'] and proof['cold_confirmation_contract'] and proof['driver_source_code_sha256']==source and proof['protocol_sha256']==V036_PROTOCOL_SHA and all(proof['checks'].values()),'Actual CUDA confirmation proof missing/changed')
        v036_require(all(v<=19800 for v in proof['projected_stage_seconds'].values()),'Ordinary lease budget proof failed')
        summary=json.loads((root/'outputs/controlled_experiments_v035/screen_summary.json').read_text());review=json.loads((root/'research/screen_review_v035.json').read_text())
        eligible=[c for c in summary['candidates'] if c['gain']>=plan['minimum_A_gain'] and c['interval']['high']<0]
        winner=min(eligible,key=lambda c:(c['MAE'],c['name'])) if eligible else None
        v036_require(review['status']=='passed' and winner is not None and winner==summary['selected_for_confirmation']==review['selected'] and winner['weights']==plan['winner_weights'] and winner['name']==plan['winner_name'],'Not the frozen lowest eligible A choice')
        scientific=json.loads((root/'research/confirmation_selection_v036.json').read_text())
        v036_require(scientific['winner']==winner and scientific['baseline_weights']==plan['baseline_weights'] and scientific['winner_weights']==plan['winner_weights'] and scientific['required_members']==plan['members'],'Frozen A scientific choice changed')
        if (out/'confirmation_result.json').exists():
            done=json.loads((out/'confirmation_result.json').read_text())
            previous=json.loads((out/'locked_before_F.json').read_text())
            v036_require(resume and done['status']=='complete' and done['source_code_sha256']==source and done['protocol_sha256']==V036_PROTOCOL_SHA and done['selection_sha256']==v036_sig(previous) and done['locked_file_sha256']==v036_sha(out/'locked_before_F.json') and done['winner_weights']==plan['winner_weights'],'Completed F result changed')
            for record in done['records'].values():
                for path,h in [(record['checkpoint_file'],record['checkpoint_sha256']),(record['report_file'],record['report_sha256'])]:v036_require(v036_sha(root/path)==h,'Completed F artifact changed')
            for artifact in done['prediction_artifacts'].values():v036_require(v036_sha(root/artifact['file'])==artifact['file_sha256'],'Completed F predictions changed')
            return done
        # Read only known inputs before writing the immutable F lock. Pressure
        # labels are loaded later, after choice/split/epochs/weights are frozen.
        known=pd.read_csv(root/'data/train.csv',usecols=['id','breath_id','R','C','time_step','u_in','u_out'],dtype={k:v for k,v in V013_DTYPES.items() if k!='pressure'})
        bids=_validate_feature_frame_v005(known);v036_require(len(bids)==67905,'Approved group count changed')
        with np.load(root/'outputs/improvements_v005/sequence/split_breath_ids.npz',allow_pickle=False) as z:
            afit=np.sort(pd.Index(bids).get_indexer(z['A_fit']));aval=np.sort(pd.Index(bids).get_indexer(z['A_validation']))
        v036_require(len(afit)==61114 and len(aval)==6791 and not np.intersect1d(afit,aval).size and np.array_equal(np.sort(np.r_[afit,aval]),np.arange(67905)),'Original A grouping changed')
        rc=known[['R','C']].to_numpy()[::80];joint=np.searchsorted([5,20,50],rc[:,0])*3+np.searchsorted([10,20,50],rc[:,1])
        from sklearn.model_selection import train_test_split
        fit,val=train_test_split(afit,test_size=6791,random_state=20261204,stratify=joint[afit]);fit=np.sort(fit);val=np.sort(val)
        v036_require(len(fit)==54323 and len(val)==6791 and not np.intersect1d(fit,val).size and not np.intersect1d(np.r_[fit,val],aval).size,'F containment/leakage')
        split=dict(seed=20261204,indices=dict(A_fit=afit.tolist(),A_validation=aval.tolist(),E_fit=fit.tolist(),E_validation=val.tolist()),breath_ids=dict(E_fit=bids[fit].tolist(),E_validation=bids[val].tolist()),group_hashes=dict(E_fit=v013_ah(bids[fit]),E_validation=v013_ah(bids[val])),counts=dict(E_fit=len(fit),E_validation=len(val)))
        v036_immutable_json(out/'F_split.json',split,resume)
        locked=dict(protocol_sha256=V036_PROTOCOL_SHA,source_code_sha256=source,winner_name=plan['winner_name'],winner_weights=plan['winner_weights'],baseline_weights=plan['baseline_weights'],members=plan['members'],input_hashes=plan['input_hashes'],component_hashes=plan['source_code_hashes'],F_split_file_sha256=v036_sha(out/'F_split.json'),train_sha256=plan['train_SHA256'],acceptance=plan['acceptance'],bootstrap_seed=plan['bootstrap_seed'],bootstrap_draws=1000,no_runner_up=True,no_F_epoch_or_weight_selection=True,cold_fit_all_members=True)
        v036_immutable_json(out/'locked_before_F.json',locked,resume);selection=v036_sig(locked)
        sources=dict(plan['source_code_hashes'],**{V036_NOTEBOOK:source,'controlled_experiments_v013.ipynb':V036_V013_SOURCE_SHA,'confirmation_protocol_v036.json':V036_PROTOCOL_SHA,'locked_before_F.json':selection})
        v013_event(out,'F_locked_before_pressure_read',selection_sha256=selection,gpu=torch.cuda.get_device_name(0),pod=os.environ.get('HOSTNAME'))
        fit_labels=v036_load_selected_labels(root/'data/train.csv',bids[fit])
        uo=known.u_out.to_numpy(copy=True).reshape(-1,80);ids=known.id.to_numpy(copy=True).reshape(-1,80)
        x,names,feature_bids=make_sequence_features(known,'combined');del known;gc.collect()
        v036_require(x.shape==(67905,80,47) and np.array_equal(feature_bids,bids),'Known feature order/schema changed')
        records={}
        for name,recipe in plan['members'].items():
            stage=out/'F_models'/name;stage.mkdir(parents=True,exist_ok=True);checkpoint=stage/'model.pt';rp=stage/'record.json'
            identity=dict(selection_sha256=selection,member_id=name,recipe=recipe,training_group_sha256=split['group_hashes']['F_fit'],source_fingerprints=sources)
            if rp.exists():
                record=json.loads(rp.read_text());v036_require(resume,'Existing member needs explicit resume');v036_verify_record(root,record,identity)
            else:
                if name not in plan['stages'][stage_key]:continue
                if stage_key=='remaining_two':
                    v036_require((out/'stage_complete_first_five.json').is_file(),'Second stage requires first-stage completion')
                    v036_require(all((out/'F_models'/m/'record.json').is_file() for m in plan['stages']['first_five']),'Second stage missing first-stage members')
                v013_event(out,'cold_F_fit_started',member=name,epochs=recipe['selected_epochs'])
                if (stage/'report.json').exists():
                    v036_require(resume,'Orphan completed report requires explicit resume');report=json.loads((stage/'report.json').read_text())
                    v036_require(report['checkpoint_sha256']==v036_sha(checkpoint),'Orphan report/model mismatch')
                else:
                    report=v036_fit_dispatch(name,recipe,x[fit],fit_labels,uo[fit],checkpoint,feature_names=names,device='cuda',resume=resume and _last_path(checkpoint).exists(),source_fingerprints=sources,epoch_log_path=stage/'epochs.jsonl')
                v036_require(report['completed'] and report['executed_epochs']==recipe['selected_epochs'] and report['training_breaths']==54323 and not report['validation_used'] and report['executed_lr_schedule']==recipe['lr_schedule'],'Cold fixed F fit incomplete/changed')
                v036_immutable_json(stage/'report.json',report,resume)
                record=dict(identity=identity,identity_sha256=v036_sig(identity),checkpoint_file=str(checkpoint.relative_to(root)),checkpoint_sha256=v036_sha(checkpoint),report_file=str((stage/'report.json').relative_to(root)),report_sha256=v036_sha(stage/'report.json'),training_breaths=54323,selected_epochs=recipe['selected_epochs'],elapsed_seconds=report['elapsed_seconds'],last_checkpoint_file=str(_last_path(checkpoint).relative_to(root)),last_checkpoint_sha256=v036_sha(_last_path(checkpoint)))
                v013_json(rp,record)
            records[name]=record;gc.collect();torch.cuda.empty_cache()
        completed_stage={m:records[m] for m in plan['stages'][stage_key] if m in records}
        v036_require(set(completed_stage)==set(plan['stages'][stage_key]),'Execution stage incomplete; preserve exact optimizer/RNG state')
        stage_result=dict(status='stage_complete',stage=stage_key,protocol_sha256=V036_PROTOCOL_SHA,source_code_sha256=source,selection_sha256=selection,records=completed_stage,no_F_validation_pressure_loaded=True,no_final_fit_or_submission=True)
        v036_immutable_json(out/('stage_complete_'+stage_key+'.json'),stage_result,resume)
        if len(records)!=len(plan['members']):return stage_result
        # Every member is a fresh completed Ffit. No validation label has been loaded.
        mask=uo[val]==0;rows=ids[val][mask];counts=mask.sum(1);predictions={};artifacts={}
        v036_immutable_array(out/'F_row_ids.npy',rows)
        for name,recipe in plan['members'].items():
            checkpoint=root/records[name]['checkpoint_file'];stage=checkpoint.parent
            p=v036_predict_dispatch(name,recipe,checkpoint,x[val],device='cuda',source_fingerprints=sources,batch_size=512)
            p=p[mask].astype(np.float64);v036_require(p.shape==rows.shape and np.isfinite(p).all(),'Nonfinite/misaligned F predictions')
            pp=stage/'F_predictions.npy';v036_immutable_array(pp,p);predictions[name]=p
            artifacts[name]=dict(file=str(pp.relative_to(root)),file_sha256=v036_sha(pp),array_sha256=v013_ah(p),row_ids_sha256=v013_ah(rows))
        grid=pressure_codebook_v016(fit_labels,uo[fit]);v036_immutable_array(out/'F_fit_pressure_codebook.npy',grid)
        baseline=selected_postprocessing_v016({k:predictions[k] for k in plan['baseline_weights']},plan['baseline_weights'],grid)
        candidate=selected_postprocessing_v016({k:predictions[k] for k in plan['winner_weights']},plan['winner_weights'],grid)
        v036_immutable_array(out/'baseline_F_predictions.npy',baseline);v036_immutable_array(out/'candidate_F_predictions.npy',candidate)
        prediction_lock=dict(selection_sha256=selection,source_code_sha256=source,protocol_sha256=V036_PROTOCOL_SHA,artifacts=artifacts,baseline_sha256=v036_sha(out/'baseline_F_predictions.npy'),candidate_sha256=v036_sha(out/'candidate_F_predictions.npy'),grid_sha256=v036_sha(out/'F_fit_pressure_codebook.npy'),row_ids_sha256=v036_sha(out/'F_row_ids.npy'),no_validation_pressure_loaded=True)
        v036_immutable_json(out/'predictions_locked_before_F_pressure.json',prediction_lock,resume)
        truth=v036_load_selected_labels(root/'data/train.csv',bids[val])[mask].astype(np.float64)
        metrics=dict(baseline=v036_metrics(baseline,truth),candidate=v036_metrics(candidate,truth));interval=v013_interval(np.abs(candidate-truth),np.abs(baseline-truth),counts,plan['bootstrap_seed'])
        gain=metrics['baseline']['mae']-metrics['candidate']['mae'];passed=bool(gain>0 and interval['high']<0)
        v036_immutable_array(out/'baseline_F_predictions.npy',baseline);v036_immutable_array(out/'candidate_F_predictions.npy',candidate)
        v036_require(v036_sha(root/plan['best_CSV'])==plan['best_CSV_SHA256'],'Official best CSV changed')
        for p,h in plan['input_hashes'].items():v036_require(v036_sha(root/p)==h,'Frozen input changed during F: '+p)
        v036_require(hashlib.sha256(v036_code(root/V036_NOTEBOOK).encode()).hexdigest()==source,'F source changed')
        result=dict(status='complete',schema='v036-F-result-v1',selection_sha256=selection,locked_file_sha256=v036_sha(out/'locked_before_F.json'),protocol_sha256=V036_PROTOCOL_SHA,source_code_sha256=source,winner_name=plan['winner_name'],winner_weights=plan['winner_weights'],baseline_weights=plan['baseline_weights'],metrics=metrics,gain=gain,interval=interval,passed=passed,records=records,prediction_artifacts=artifacts,F_scored_points=len(rows),F_truth_array_sha256=v013_ah(truth),F_row_ids_array_sha256=v013_ah(rows),source_fingerprints=sources,cold_fit_all_members=True,no_F_epoch_or_weight_selection=True,original_A_validation_excluded=True,confirmation_is_resampling_evidence=True,no_runner_up_after_failure=True,final_fit_performed=False,submission_performed=False,official_best_preserved=.17427,elapsed_seconds=time.monotonic()-began,completed_utc=datetime.now(timezone.utc).isoformat())
        v036_immutable_json(out/'confirmation_result.json',result,resume=False)
        v013_event(out,'confirmation_complete',passed=passed,baseline_MAE=metrics['baseline']['mae'],candidate_MAE=metrics['candidate']['mae'],gain=gain)
        return result
    finally:
        for f in reversed(locks):fcntl.flock(f,fcntl.LOCK_UN);f.close()

if os.environ.get('PA3_RUN_V036','0')=='1':
    V036_RESULT=run_confirmation_v036(Path.cwd(),resume=os.environ.get('PA3_RESUME_V036','0')=='1',stage_key=os.environ['PA3_V036_STAGE'])
    print(json.dumps({k:v for k,v in V036_RESULT.items() if k not in ['records','prediction_artifacts','source_fingerprints']},indent=2))
